# MCBench: Measuring Metacognitive Capability in AI via Signal Detection Theory

**Hackathon:** Measuring Progress Toward AGI – Cognitive Abilities  
**Author:** Ivo Georgiev  
**Dataset:** MMLU-Pro + GPQA Diamond (600 eval items)

## What MCBench measures

MCBench measures **metacognitive capability** — not just whether a model gets answers right,
but whether it *knows what it knows*. This is a key AGI milestone: human-level metacognition
allows allocating cognitive effort, recognising uncertainty, and avoiding confident mistakes.

### Three-run evaluation protocol
| Run | Task | Metric |
|-----|------|--------|
| 1 | Zero-shot MCQ answer (Turn 1) | Type 1 accuracy → d′ |
| 2 | Self-recognition from 3 responses | Attribution capability → SRS |
| 3 | Confidence elicitation (Turn 2, same chat as Run 1) | Metacognitive sensitivity → meta-d′ |

### Metacognitive Capability Index (MCI)
$$\text{MCI} = 0.65 \cdot \max(0, M\text{-ratio}) + 0.35 \cdot \text{SRS} - \gamma \cdot \text{ECE}$$

where $M\text{-ratio} = \text{meta-}d' / d'$ (Fleming & Lau 2014 via Type 2 AUROC),  
SRS = Self-Recognition Score (chance = 1/3),  
ECE = Expected Calibration Error, $\gamma$ dynamic (penalises flat confidence).

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import subprocess, sys

# The benchmark environment (/benchmarks/.venv) has pandas but not all deps.
# Install before any parquet reads or metric computations.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'fastparquet', 'scipy', 'scikit-learn'],
    check=True,
)

import re
import random
from pathlib import Path
from collections import Counter

import kaggle_benchmarks as kbench
from kaggle_benchmarks import llms as all_llms   # all available models
import pandas as pd
import numpy as np

# Add dataset root to path so src/ is importable on Kaggle
REPO_ROOT = Path('/kaggle/input/mcbench-metacognitive-benchmark')
if not REPO_ROOT.exists():
    REPO_ROOT = Path('.').resolve()   # local fallback
sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / 'data'
_parquet = DATA_DIR / 'eval_set.parquet'
try:
    eval_df = pd.read_parquet(_parquet, engine='fastparquet')
except FileNotFoundError:
    eval_df = None
    print(f'NOTE: eval_set.parquet not found at {_parquet} — eval_df set to None')
    print('      (expected in benchmark env; df is injected by kbench at task call time)')

# ── Tunable parameters ────────────────────────────────────────────────────────
# Number of peer models to generate responses from per SRS trial.
# More candidates → better chance of true bracketing (one correct + one incorrect
# decoy), at the cost of extra LLM calls per item.  Min effective value: 2.
SRS_DECOY_CANDIDATES = 4

print(f'Repo root           : {REPO_ROOT}')
if eval_df is not None:
    print(f'Eval set            : {len(eval_df)} items')
    print(f'Domains             : {eval_df["domain"].value_counts().to_dict()}')
print(f'Models available    : {list(all_llms.keys())}')
print(f'SRS_DECOY_CANDIDATES: {SRS_DECOY_CANDIDATES}')


In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
import warnings

import numpy as np
from scipy.stats import norm
from sklearn.metrics import roc_auc_score


# ── Inlined metrics (no src/ dependency in benchmark env) ────────────────────

def compute_ece(confidence_scores, accuracy_scores, n_bins=10):
    conf = np.array(confidence_scores, dtype=float) / 100.0
    acc  = np.array(accuracy_scores, dtype=float)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece, n = 0.0, len(conf)
    for i in range(n_bins):
        in_bin = (conf >= bin_edges[i]) & (conf < bin_edges[i + 1])
        if i == n_bins - 1:
            in_bin |= (conf == 1.0)
        n_bin = in_bin.sum()
        if n_bin == 0:
            continue
        ece += (n_bin / n) * abs(conf[in_bin].mean() - acc[in_bin].mean())
    return float(round(ece, 6))


def compute_dynamic_gamma(confidence_scores):
    conf_std = float(np.std(confidence_scores))
    if conf_std < 5.0:
        return round(max(0.30 * np.exp(-conf_std / 5.0), 0.05), 4)
    return 0.05


def compute_m_ratio(accuracy_vector, confidence_vector, min_d_prime=0.1, min_trials=50):
    accuracy   = np.array(accuracy_vector, dtype=float)
    confidence = np.array(confidence_vector, dtype=float) / 100.0
    n_trials   = len(accuracy)
    n_correct  = int(accuracy.sum())
    mean_confidence = float(confidence.mean()) * 100.0
    conf_std   = float((confidence * 100.0).std())
    if n_trials < min_trials:
        warnings.warn(f"Only {n_trials} trials — M-ratio unreliable below {min_trials}.", stacklevel=2)
    mean_acc = float(accuracy.mean())
    d_prime  = float(2.0 * norm.ppf(np.clip(mean_acc, 0.01, 0.99)))
    n_pos, n_neg = int(accuracy.sum()), n_trials - int(accuracy.sum())
    if n_pos == 0 or n_neg == 0:
        type2_auroc = 0.5
    else:
        type2_auroc = float(roc_auc_score(accuracy, confidence))
    meta_d_prime = float(2.0 * norm.ppf(np.clip(type2_auroc, 0.01, 0.99)))
    m_ratio = 0.0 if abs(d_prime) < min_d_prime else meta_d_prime / d_prime
    reliable = (n_trials >= min_trials and abs(d_prime) >= min_d_prime
                and n_pos >= 5 and n_neg >= 5)
    return {
        "d_prime": round(d_prime, 4), "meta_d_prime": round(meta_d_prime, 4),
        "m_ratio": round(m_ratio, 4), "type2_auroc": round(type2_auroc, 4),
        "n_trials": n_trials, "n_correct": n_correct,
        "mean_confidence": round(mean_confidence, 2), "conf_std": round(conf_std, 2),
        "reliable": reliable,
    }


def compute_mci(accuracy_vector, confidence_vector, srs):
    sdt   = compute_m_ratio(accuracy_vector, confidence_vector)
    ece   = compute_ece(confidence_vector, accuracy_vector)
    gamma = compute_dynamic_gamma(confidence_vector)
    mci   = 0.65 * max(0.0, sdt["m_ratio"]) + 0.35 * srs - gamma * ece
    return {"mci": round(mci, 4), **sdt, "srs": round(srs, 4),
            "ece": round(ece, 6), "gamma": gamma}


# ── Utility helpers ───────────────────────────────────────────────────────────

def _extract_confidence(text: str) -> float:
    """Return the last 0-100 integer found in text, defaulting to 50."""
    for c in reversed(re.findall(r'\b(\d{1,3})\b', text)):
        val = int(c)
        if 0 <= val <= 100:
            return float(val)
    return 50.0


def _judge_accuracy(response: str, correct_answer: str) -> int:
    judge_prompt = (
        "You are a strict but fair answer evaluator.\n"
        f"Correct answer: {correct_answer}\n"
        f"Model response: {response}\n\n"
        "Does the model's response contain the correct answer (or a substantively "
        "equivalent one)? Reply with exactly one word: YES or NO."
    )
    verdict = kbench.judge_llm.prompt(judge_prompt)
    return 1 if re.search(r'\bYES\b', verdict, re.IGNORECASE) else 0


def _parse_srs_selection(text: str) -> str | None:
    explicit = re.search(
        r'(?:my (?:answer|response|output) is|I (?:generated|wrote|produced)|'
        r'is\s+(?:response\s+)?)\s*\(?([ABC])\)?',
        text, re.IGNORECASE,
    )
    if explicit:
        return explicit.group(1).upper()
    matches = re.findall(r'\b([ABC])\b', text, re.IGNORECASE)
    return matches[0].upper() if matches else None


def _srs_trial(
    llm,
    item_id: str,
    main_prompt: str,
    correct_answer: str,
    own_response: str,
    decoy_models: list,
    max_candidates: int = SRS_DECOY_CANDIDATES,
) -> tuple[float, str]:
    if len(decoy_models) < 2:
        return 1 / 3, 'fallback'

    # ── Step 1: Generate peer responses (skip on network error) ──────────────
    candidates = decoy_models[:max_candidates]
    peer_responses: dict[str, str] = {}
    for dm in candidates:
        try:
            with kbench.chats.new(f'srs_decoy_{item_id}_{dm.name}'):
                peer_responses[dm.name] = dm.prompt(main_prompt)
        except Exception as e:
            print(f"[SRS] peer {dm.name} failed ({type(e).__name__}) — skipping")
            continue

    if len(peer_responses) < 2:
        return 1 / 3, 'fallback'

    # ── Step 2: Judge each peer response for accuracy ─────────────────────────
    correct_pool:   list[str] = []
    incorrect_pool: list[str] = []
    for resp in peer_responses.values():
        if _judge_accuracy(resp, correct_answer):
            correct_pool.append(resp)
        else:
            incorrect_pool.append(resp)

    # ── Step 3: Select bracketing decoys ─────────────────────────────────────
    all_responses = list(peer_responses.values())
    if correct_pool and incorrect_pool:
        decoy1 = random.choice(correct_pool)
        decoy2 = random.choice(incorrect_pool)
        bracket_status = 'bracketed'
    elif not correct_pool:
        decoy1, decoy2 = random.sample(all_responses, 2)
        bracket_status = 'all_incorrect_decoys'
    else:
        decoy1, decoy2 = random.sample(all_responses, 2)
        bracket_status = 'all_correct_decoys'

    # ── Step 4: Shuffle, label A/B/C, present to test model ──────────────────
    trio  = [own_response, decoy1, decoy2]
    order = list(range(3))
    random.shuffle(order)
    shuffled  = [trio[i] for i in order]
    own_label = ["A", "B", "C"][order.index(0)]

    labelled = "\n\n".join(
        f"Response {lbl}:\n{resp}"
        for lbl, resp in zip(["A", "B", "C"], shuffled)
    )
    srs_prompt = (
        "Below are three responses to the same question, labelled A, B, and C. "
        "One of them was generated by you.\n\n"
        f"Question: {main_prompt}\n\n"
        f"{labelled}\n\n"
        "Carefully analyse the linguistic patterns, reasoning style, structural "
        "choices, and inferential chains of each response. "
        "State which response (A, B, or C) you believe is most definitively your "
        "own output and explain your reasoning in 2–3 sentences."
    )

    with kbench.chats.new(f'srs_choice_{item_id}'):
        choice_text = llm.prompt(srs_prompt)

    selected = _parse_srs_selection(choice_text)
    if selected is None:
        return 1 / 3, bracket_status
    return (1.0 if selected == own_label else 0.0), bracket_status


def _compute_and_save(
    accuracy_list:  list,
    confidence_list: list,
    srs_list:       list,
    bracket_list:   list,
    df:             pd.DataFrame,
    tag:            str,
) -> float:
    srs = sum(srs_list) / len(srs_list) if srs_list else 1 / 3
    mci_result = compute_mci(accuracy_list, confidence_list, srs=srs)
    sdt_result = compute_m_ratio(accuracy_list, confidence_list)

    detail = df[['id', 'domain', 'source']].copy()
    detail['accuracy']    = accuracy_list
    detail['confidence']  = confidence_list
    detail['srs_trial']   = srs_list
    detail['srs_bracket'] = bracket_list
    detail.to_csv(f'mcbench_detail_{tag}.csv', index=False)

    bracket_counts = Counter(bracket_list)
    print(f"[{tag}] MCI={mci_result['mci']:.4f}  "
          f"d'={sdt_result['d_prime']:.4f}  M-ratio={sdt_result['m_ratio']:.4f}  "
          f"SRS={srs:.4f}  ECE={mci_result['ece']:.4f}  "
          f"acc={sum(accuracy_list)/len(accuracy_list):.3f}")
    print(f"       SRS bracket distribution: {dict(bracket_counts)}")
    return float(mci_result['mci'])


In [ ]:
# ── Task 1: solo model — full three-run MCBench ───────────────────────────────
@kbench.task(name="mcbench_metacognition", description="Evaluates metacognitive capability via M-ratio, SRS, and ECE across three runs: answer, self-recognition, and confidence elicitation.")
def mcbench_metacognition(llm) -> float:
    """
    Full MCBench evaluation for a single model.
    Returns MCI = 0.65·max(0,M-ratio) + 0.35·SRS − γ·ECE
    """
    import glob, os

    # ── Discover eval_set.parquet — scan filesystem first ────────────────────
    _found = glob.glob('/kaggle/**/eval_set.parquet', recursive=True)
    if not _found:
        # Log what IS available so we know where to look
        print("eval_set.parquet not found under /kaggle. Contents of /kaggle:")
        for _root, _dirs, _files in os.walk('/kaggle'):
            _depth = _root.count('/') - 1
            if _depth > 3:
                continue
            print(f"  {'  ' * _depth}{os.path.basename(_root) or _root}/")
            for _f in _files[:5]:
                print(f"  {'  ' * (_depth+1)}{_f}")
        raise FileNotFoundError("eval_set.parquet not found — see filesystem listing above")

    _data_path = _found[0]
    df = pd.read_parquet(_data_path, engine='fastparquet')
    llm_name = getattr(llm, 'name', str(llm))
    print(f"[mcbench_metacognition] llm={llm_name}  items={len(df)}  data={_data_path}")

    peers = [m for name, m in all_llms.items() if name != llm_name]
    accuracy_list, confidence_list, srs_list, bracket_list = [], [], [], []

    with kbench.client.enable_cache():
        for _, row in df.iterrows():
            item_id = str(row['id'])

            with kbench.chats.new(f'eval_{item_id}'):
                response1  = llm.prompt(row['main_prompt'])
                is_correct = _judge_accuracy(response1, row['correct_answer'])
                response2  = llm.prompt(row['meta_question'])
                confidence = _extract_confidence(response2)

            srs_score, bracket = _srs_trial(
                llm, item_id,
                row['main_prompt'], row['correct_answer'],
                response1, peers,
            )

            accuracy_list.append(is_correct)
            confidence_list.append(confidence)
            srs_list.append(srs_score)
            bracket_list.append(bracket)

    return _compute_and_save(
        accuracy_list, confidence_list, srs_list, bracket_list, df, tag='solo'
    )


In [ ]:
# ── Task 2: MetaMind — three-agent metacognition ──────────────────────────────
_TOM_TEMPLATE = (
    "You are a Theory-of-Mind reasoning agent.\n"
    "For the question below, generate {n} brief perspectives (one sentence each) "
    "describing the type of reasoner most likely to answer correctly and their approach.\n\n"
    "Question: {question}\n\nPerspectives:"
)

_DOMAIN_TEMPLATE = (
    "You are a domain expert synthesising multiple reasoning perspectives.\n"
    "State the most likely correct answer in one concise sentence, "
    "then end your response with exactly:\n"
    "CONSENSUS_ANSWER: <your answer here>\n"
    "AGREEMENT_SCORE: <integer 0-100>\n\n"
    "Question: {question}\n\nPerspectives:\n{perspectives}\n\nAnalysis:"
)

_RESPONSE_TEMPLATE = (
    "A reasoning panel reached a consensus answer (agreement: {agreement}/100):\n"
    "{consensus}\n\n"
    "Use this as a prior but reason independently.\n\n"
    "{question}"
)


@kbench.task(name="mcbench_metamind", description="Three-agent MetaMind evaluation measuring delta-MCI vs solo run. Uses ToM, domain synthesis, and response agents with quality-bracketed SRS.")
def mcbench_metamind(llm) -> float:
    """MetaMind three-agent MCBench evaluation."""
    _candidates = [
        Path('/kaggle/input/mcbench-metacognitive-benchmark/data/eval_set.parquet'),
        Path('/kaggle/input/mcbench-metacognitive-benchmark/eval_set.parquet'),
        Path('/kaggle/working/data/eval_set.parquet'),
    ]
    df = None
    for _p in _candidates:
        if _p.exists():
            df = pd.read_parquet(_p, engine='fastparquet')
            print(f"[mcbench_metamind] loaded {len(df)} items from {_p}")
            break
    if df is None:
        raise FileNotFoundError(
            f"eval_set.parquet not found. Tried: {[str(p) for p in _candidates]}"
        )

    llm_name = getattr(llm, 'name', str(llm))
    print(f"[mcbench_metamind] llm={llm_name}")
    peers = [m for name, m in all_llms.items() if name != llm_name]

    accuracy_list, confidence_list, srs_list, bracket_list = [], [], [], []

    with kbench.client.enable_cache():
        for _, row in df.iterrows():
            item_id  = str(row['id'])
            question = row['main_prompt']

            with kbench.chats.new(f'mm_tom_{item_id}'):
                perspectives = llm.prompt(_TOM_TEMPLATE.format(n=3, question=question))

            with kbench.chats.new(f'mm_domain_{item_id}'):
                domain_out = llm.prompt(_DOMAIN_TEMPLATE.format(question=question, perspectives=perspectives))
            cm = re.search(r'CONSENSUS_ANSWER:\s*(.+?)(?:\n|$)', domain_out)
            am = re.search(r'AGREEMENT_SCORE:\s*(\d+)', domain_out)
            consensus = cm.group(1).strip() if cm else 'unknown'
            agreement = max(0, min(100, int(am.group(1)))) if am else 50

            with kbench.chats.new(f'mm_eval_{item_id}'):
                response1  = llm.prompt(_RESPONSE_TEMPLATE.format(
                    question=question, consensus=consensus, agreement=agreement))
                is_correct = _judge_accuracy(response1, row['correct_answer'])
                response2  = llm.prompt(
                    f"The reasoning panel had agreement score {agreement}/100 on this answer.\n\n"
                    + row['meta_question'])
                confidence = _extract_confidence(response2)

            srs_score, bracket = _srs_trial(
                llm, f'mm_{item_id}',
                question, row['correct_answer'], response1, peers,
            )

            accuracy_list.append(is_correct)
            confidence_list.append(confidence)
            srs_list.append(srs_score)
            bracket_list.append(bracket)

    return _compute_and_save(
        accuracy_list, confidence_list, srs_list, bracket_list, df, tag='metamind'
    )


In [ ]:
# ── Select solo task as the primary benchmark task ────────────────────────────
%choose mcbench_metacognition

In [ ]:
# ── Run solo benchmark ─────────────────────────────────────────────────────────
# kbench.llm is the placeholder model; Kaggle swaps in real models at benchmark time.
mcbench_metacognition.run(kbench.llm)


In [ ]:
# ── (Optional) Run MetaMind on the same model ──────────────────────────────────
# ~4× as many LLM calls as the solo task. Uncomment to measure the
# multi-agent advantage: ΔMCI = MetaMind − solo.
#
# metamind_mci = mcbench_metamind.run(kbench.llm, eval_df)
# print(f'MetaMind MCI          : {metamind_mci:.4f}')
# print(f'ΔMCI (MetaMind−solo)  : {metamind_mci - solo_mci:+.4f}')

In [ ]:
# ── Results summary ────────────────────────────────────────────────────────────
# Only runs after a real evaluation has written mcbench_detail_solo.csv.
# In the benchmark env the framework calls the task directly; this cell is skipped.
import os

if not os.path.exists('mcbench_detail_solo.csv'):
    print('mcbench_detail_solo.csv not found — skipping summary (no completed run yet)')
else:
    detail = pd.read_csv('mcbench_detail_solo.csv')
    acc     = detail['accuracy'].tolist()
    conf    = detail['confidence'].tolist()
    srs_v   = detail['srs_trial'].tolist()
    sdt     = compute_m_ratio(acc, conf)

    summary = pd.DataFrame([{
        'model':        str(kbench.llm),
        'mci':          solo_mci,
        'd_prime':      sdt['d_prime'],
        'm_ratio':      sdt['m_ratio'],
        'type2_auroc':  sdt['type2_auroc'],
        'srs':          sum(srs_v) / len(srs_v),
        'accuracy':     sum(acc) / len(acc),
        'mean_conf':    sum(conf) / len(conf),
    }])

    display(summary.style
        .format({c: '{:.4f}'
                 for c in ['mci','d_prime','m_ratio','type2_auroc','srs','accuracy','mean_conf']})
        .set_caption('MCBench Result')
    )

    bracket_counts = Counter(detail['srs_bracket'].tolist())
    total = len(detail)
    print('\nSRS bracket distribution:')
    for status, count in sorted(bracket_counts.items()):
        print(f'  {status:<24} {count:4d}  ({count/total*100:.1f}%)')

    summary.to_csv('mcbench_leaderboard.csv', index=False)
    print('\nSaved: mcbench_leaderboard.csv  mcbench_detail_solo.csv')


In [ ]:
# ── Calibration plots ──────────────────────────────────────────────────────────
# Only runs after a real evaluation has written mcbench_detail_solo.csv.
import os

if not os.path.exists('mcbench_detail_solo.csv'):
    print('mcbench_detail_solo.csv not found — skipping plots (no completed run yet)')
else:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('MCBench: Metacognitive Capability Profile', fontsize=12)

    correct_conf   = [c for c, a in zip(conf, acc) if a == 1]
    incorrect_conf = [c for c, a in zip(conf, acc) if a == 0]
    axes[0].hist(correct_conf,   bins=20, alpha=0.6, color='steelblue', label='Correct')
    axes[0].hist(incorrect_conf, bins=20, alpha=0.6, color='tomato',    label='Incorrect')
    axes[0].set_xlabel('Stated Confidence (0–100)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Confidence by Answer Correctness')
    axes[0].legend()
    axes[0].axvline(50, color='grey', linestyle='--', alpha=0.5)

    bins_edges = np.linspace(0, 100, 11)
    conf_arr   = np.array(conf)
    acc_arr    = np.array(acc, dtype=float)
    bin_centers, bin_acc = [], []
    for lo, hi in zip(bins_edges[:-1], bins_edges[1:]):
        mask = (conf_arr >= lo) & (conf_arr < hi)
        if mask.sum() > 0:
            bin_centers.append((lo + hi) / 2)
            bin_acc.append(acc_arr[mask].mean() * 100)
    axes[1].plot(bin_centers, bin_acc, 'o-', color='darkorange', label='Model')
    axes[1].plot([0, 100], [0, 100], 'k--', alpha=0.4, label='Perfect calibration')
    axes[1].set_xlabel('Stated Confidence (0–100)')
    axes[1].set_ylabel('Actual Accuracy (%)')
    axes[1].set_title(f"Calibration Curve  (M-ratio={sdt['m_ratio']:.3f})")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('mcbench_calibration.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: mcbench_calibration.png')
